# Lab 09: Transformer 與預訓練中文 BERT 模型

在本單元中，我們將接觸當今 AI 革命的幕後功臣 — **Transformer 架構**，並學習如何調用強大的預訓練模型（Pre-trained Models）。
我們將使用 Hugging Face `transformers` 套件，載入中文的 **BERT 模型**，進行「克漏字填空 (Masked LM)」與「中文社群貼文的情緒分類」推理！

### 環境安全網
安裝 Hugging Face transformers 套件並確認版本。
BERT 模型約 400MB，首次載入需要一段時間，請耐心等待。

In [ ]:
import subprocess, sys

# 安裝 transformers
print('正在安裝/確認 transformers 套件...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers', '-q'])

import transformers, torch
print(f'transformers: {transformers.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print()
print('準備就緒！接下來會下載中文 BERT 模型 (約 400MB)')
print('首次執行會比較慢，之後 Colab 會快取，重新執行就很快了。')


### 任務 1: 安裝 Hugging Face 套件
在 Google Colab 中需要先下載安裝 `transformers` 套件。

In [ ]:
# 上方「環境安全網」已自動安裝，這裡示範在 Colab 手動安裝套件的指令
# (驚嘆號 ! 代表在終端機 shell 執行；-q 表示安靜模式，減少輸出)
!pip install transformers -q

### 延伸探索 1: Tokenizer 拆解 — 文字如何被 BERT 讀懂？

BERT 在處理文字之前，會先用 **Tokenizer (分詞器)** 將句子切成子詞單位，並轉換為整數 ID。讓我們親眼看看這個過程：


In [ ]:
from transformers import BertTokenizer

# 載入 BERT 中文分詞器
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

test_sentence = '今天的 AI 課程讓我大開眼界！'
print('原始句子:', test_sentence)

# 分詞：將句子切成子詞 (subword) 單位
tokens = tokenizer.tokenize(test_sentence)
print('\nTokenizer 切出的子詞:', tokens)
print('子詞數量:', len(tokens))

# 轉換為 ID 序列 (模型實際看到的數字)
input_ids = tokenizer.encode(test_sentence)
print('\n轉換為 ID 序列:', input_ids)

# 解碼回文字 (驗證可逆)
decoded = tokenizer.decode(input_ids)
print('從 ID 解碼回來:', decoded)

# (可自由修改) 換一個你想測試的句子，觀察 BERT 如何分詞
# 例如：tokenizer.tokenize("你好世界，這是我的第一個 AI 模型")


### 任務 2: BERT 克漏字預測 (Masked Language Modeling)
BERT 是透過「蓋住一部分的字，並讓模型預測被蓋住的字」來學習中文文法與詞彙語意的。這個任務稱為 Masked LM。
我們將載入繁體/簡體通用的 `bert-base-chinese` 模型，看看它多有默契！

In [ ]:
from transformers import pipeline

print("正在載入中文 BERT 克漏字預測模型...")
# 使用 pipeline 工具快速載入
unmasker = pipeline("fill-mask", model="bert-base-chinese")
print("模型載入成功！")

請嘗試讓 BERT 猜測被 `[MASK]` 遮蓋住的字是甚麼。我們執行下方儲存格：

In [ ]:
sentence = "今天天氣很好，我想去[MASK]玩。"
results = unmasker(sentence)

print(f"原始句子: {sentence}\n")
print("BERT 的前 5 個預測結果:")
for i, res in enumerate(results):
    print(f"排名 {i+1}: 預測字 -> '{res['token_str']}' (信賴度: {res['score']*100:.2f}%)")

**動手修改挑戰**：請修改下方句子中的字，並將其中一個字替換成 `[MASK]`，看看 BERT 能不能猜出來！

In [ ]:
# TODO: 寫一句中文，並把你想讓 BERT 猜的那個字換成 [MASK]
# 例如：my_sentence = "我每天早上都會喝一杯[MASK]。"
# ??? (請在此填寫你的答案)

my_results = unmasker(my_sentence)
print("你的句子:", my_sentence)
for i, res in enumerate(my_results):
    print(f"預測 {i+1}: {res['sequence']} (機率: {res['score']*100:.2f}%)")

### 任務 3: 中文情緒分析 (Sentiment Analysis)
現在，我們載入一個經過「微調 (Fine-tuned)」的情緒分析 BERT 變體模型，它可以看懂中文句子，並分析出它是正面情緒還是負面情緒。

In [ ]:
print("正在載入中文情緒分類模型...")
# 載入適用於中文情緒分類的模型 (基於京東商城評論微調的 RoBERTa)
classifier = pipeline("sentiment-analysis", model="uer/roberta-base-finetuned-jd-binary-chinese")
print("分類模型載入成功！")

我們測試兩句情緒截然不同的話，印出分類標籤與模型信心分數。
此模型 (`uer/roberta-base-finetuned-jd-binary-chinese`) 的輸出標籤為：
- `positive (stars 4 and 5)`：正面 (Positive)
- `negative (stars 1, 2 and 3)`：負面 (Negative)

In [ ]:
def analyze_chinese_sentiment(text):
    res = classifier(text)[0]
    # 此模型的標籤字串為 "positive (stars 4 and 5)" / "negative (stars 1, 2 and 3)"
    # 用 'positive' 是否出現在標籤中來判斷，並相容其他版本可能回傳的 LABEL_1
    label_str = res['label'].lower()
    is_positive = 'positive' in label_str or label_str in ('label 1', 'label_1')
    label = "正面情緒 (Positive)" if is_positive else "負面情緒 (Negative)"
    score = res['score']
    print(f"文字: 『{text}』")
    print(f"預測結果: {label} (信賴度: {score*100:.2f}%)\n")

analyze_chinese_sentiment("這堂深度學習課也太好玩了吧！")
analyze_chinese_sentiment("電腦教室冷氣不夠冷，排骨便當又好鹹，真不開心。")

**動手修改挑戰**：請在下方隨意輸入幾句中文社群貼文（例如最近的心情、發牢騷或是讚美的話），測試看看 BERT 的情緒天線準不準！

In [ ]:
# (可自由修改) 替換下面的句子，測測看 BERT 的情緒天線準不準
analyze_chinese_sentiment("數學考試寫不完，真的快要崩潰了。")
analyze_chinese_sentiment("這張顯示卡的效能跟怪物一樣強。")

### 延伸探索 2: 遷移學習概念圖解

```
遷移學習三步驟
==============================================================

步驟 1: 預訓練 (Pre-training)  [Google/Meta/Microsoft 花費百萬美元完成]
  大量文字資料 (整個維基百科 + Common Crawl)
    ↓ 克漏字填空 (MLM) + 下一句預測 (NSP) 任務
  BERT 模型學會深層中文語法與語意

步驟 2: 微調 (Fine-tuning)     [你/我/小公司 花費幾小時即可完成]
  少量標記資料 (如 1000 筆電影評論)
    ↓ 在 BERT 頂端加一個分類層，繼續訓練
  模型從「通用語意理解」特化為「情感分類」

步驟 3: 推理 (Inference)       [你的 App 上線後即時使用]
  新的使用者輸入文字
    ↓ 通過微調後的模型
  輸出：正面 / 負面情緒預測

關鍵優勢：你不需要從頭訓練！巨頭已經幫你把最難的部分完成了。
==============================================================
```

在 Lab 中，我們使用的 `uer/roberta-base-finetuned-jd-binary-chinese`就是一個已經被**微調**過的模型，專門用於中文電商評論的情感分析。


---

## Lab 09 學習重點小結
### 核心概念掌握
- Transformer 的核心是**自注意力機制 (Self-Attention)**：讓句子中每個詞動態關注其他詞。
- BERT 的預訓練任務：克漏字填空 (MLM) + 下一句預測 (NSP)，讓模型學習深層語意。
- 遷移學習：下載預訓練好的大模型，只需少量資料微調即可解決新任務。
- Hugging Face pipeline 封裝了 tokenize → 推理 → decode 全流程，三行程式碼即可使用頂級模型。

### 快速自評測驗

**Q1. BERT 只使用了 Transformer 的哪一部分？**
- A. 解碼器 (Decoder)
- B. 編碼器 (Encoder)
- C. 編碼器 + 解碼器

<details><summary>查看解答</summary>

**答案：B — BERT 是純 Encoder 架構，專長於文字理解任務；GPT 系列則使用純 Decoder。**

</details>

**Q2. 遷移學習 (Transfer Learning) 的優勢是？**
- A. 必須從頭訓練模型
- B. 利用已有知識，用少量資料快速解決新任務
- C. 不需要任何訓練資料

<details><summary>查看解答</summary>

**答案：B — 預訓練模型已學習通用語言知識，微調只需少量標記資料即可達到高精度。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「Transformer 的核心是自注意力機制 (Self-Attention)」
- [ ] 我可以用一句話解釋「BERT 的預訓練任務」
- [ ] 我可以用一句話解釋「遷移學習」
- [ ] 我的程式碼成功執行並得到預期輸出
